# Notebook 01 — BOQ Parser: Understanding Raw Data

## What This Notebook Does

This is **Layer 1** of the RateIQ pipeline — the raw data ingestion and exploration layer.

We have **70+ Excel files** in `data/raw/`. These are NOT one file per project. Instead, each file is either:
- A **complete project BOQ** (e.g. `BOQ - 01.xlsx`) containing multiple trade sheets
- A **single work-category sheet** (e.g. `HVAC.xlsx`, `Civil & ID BOQ.xlsx`) extracted from a project
- A **variation order** documenting scope changes to an original BOQ

Our goal: **parse ALL files, extract every line item that has a rate, and produce a single clean CSV** that will feed into the RAG knowledge base in Notebook 02.

## What is Layer 1?

Layer 1 is purely about **data extraction and cleaning** — no embeddings, no LLMs yet. We need to understand the raw data completely before we can build anything on top of it. A junior developer reading this notebook should be able to understand every file structure, every cleaning decision, and why it was made.

## The 4 Key Questions We Answer Here

1. **How many files do we have and what types?** — Inventory + auto-categorization by filename
2. **What is the raw structure of each file?** — Sheet inventory, header positions, merged cells, junk rows
3. **How do we reliably extract line items with rates?** — Header detection, row classification, column normalization
4. **What is the final cleaned dataset?** — Shape, coverage, rate statistics, distribution by work category

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json, warnings, re
from pprint import pprint
warnings.filterwarnings('ignore')

# Load env
from dotenv import load_dotenv
load_dotenv()

# Paths
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

print(f"pandas: {pd.__version__}")
print(f"Files in data/raw: {len(list(RAW_DIR.glob('*.xlsx'))) + len(list(RAW_DIR.glob('*.xls')))}")

pandas: 3.0.2
Files in data/raw: 70


## Step 1 — File Inventory

First let's take inventory of all files and categorize them by type based on their filename.

We assign each file to one of these categories using simple keyword matching on the filename:

| Category | Meaning |
|---|---|
| `complete_boq` | Full project BOQ with multiple trades |
| `civil_id` | Civil works, interior design, finishes |
| `hvac` | Heating, ventilation, air conditioning |
| `electrical` | Electrical works, switchgear, wiring |
| `plumbing` | Plumbing and sanitary works |
| `fire_fighting` | Fire alarm, suppression systems |
| `elv_systems` | Extra-low voltage: CCTV, data, PA, sound |
| `variation` | Variation orders (scope changes) |
| `special_works` | Woodwork, facade, stairs, addenda |
| `other` | Anything that doesn't match above |

In [2]:
def categorize_file(filename):
    """Assign a work category to a file based on its name."""
    f = filename.lower()
    if 'boq' in f:
        return 'complete_boq'
    elif any(x in f for x in ['civil', 'flooring', 'ceiling', 'carpentry', 'painting', 'glass']):
        return 'civil_id'
    elif any(x in f for x in ['hvac']):
        return 'hvac'
    elif any(x in f for x in ['electric', 'switchgear', 'wiring', 'cable', 'earthing', 'fitting']):
        return 'electrical'
    elif 'plumbing' in f:
        return 'plumbing'
    elif any(x in f for x in ['fire', 'fa.xlsx']):
        return 'fire_fighting'
    elif any(x in f for x in ['cctv', 'data', 'pa &', 'sound', 'shop']):
        return 'elv_systems'
    elif 'variation' in f:
        return 'variation'
    elif any(x in f for x in ['wood', 'wardrobe', 'window', 'stair', 'addendum']):
        return 'special_works'
    return 'other'


# Scan all Excel files, skipping Windows Zone.Identifier metadata
files = [
    f for f in RAW_DIR.glob('*')
    if f.suffix in ['.xlsx', '.xls']
    and 'Zone.Identifier' not in f.name
]

inventory = []
for f in sorted(files):
    inventory.append({
        'filename': f.name,
        'category': categorize_file(f.name),
        'size_kb': round(f.stat().st_size / 1024, 1)
    })

df_inventory = pd.DataFrame(inventory)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 60)
print(df_inventory.to_string(index=False))
print("\n" + "="*60)
print("FILES PER CATEGORY:")
print(df_inventory['category'].value_counts().to_string())

                            filename      category  size_kb
                 1. CIVIL WORKS.xlsx      civil_id    381.4
                        1.Civil.xlsx      civil_id    295.1
                    2. FLOORING.xlsx      civil_id    382.0
                     3. CEILING.xlsx      civil_id    380.8
                   4. CARPENTRY.xlsx      civil_id    384.1
             5. GLASS AND METAL.xlsx      civil_id    382.3
                    6. PAINTING.xlsx      civil_id    379.8
              Addendum (Facade).xlsx special_works    387.5
                      BOQ - 01 .xlsx  complete_boq    381.2
                      BOQ - 02 .xlsx  complete_boq     44.3
                      BOQ - 03 .xlsx  complete_boq     17.7
                      BOQ - 05 .xlsx  complete_boq    560.9
                       BOQ - 12.xlsx  complete_boq   2432.3
                      BOQ - 14 .xlsx  complete_boq    444.5
                         BOQ-06.xlsx  complete_boq     27.7
                           CCTV.xlsx   e

## Step 2 — Sheet Inventory

Now let's look at what sheets are **INSIDE** each file. Even though files are already split by category, some files have multiple sheets — for example a complete BOQ might have separate sheets for Civil, HVAC, and Electrical.

We need this inventory to know:
- Which files are single-sheet vs multi-sheet
- The row/column dimensions of each sheet (helps us skip tiny cover/summary pages)
- Whether we need `openpyxl` (`.xlsx`) or `xlrd` (`.xls`) as the engine

In [3]:
sheet_inventory = []

for f in sorted(files):
    engine = 'xlrd' if f.suffix == '.xls' else 'openpyxl'
    try:
        xl = pd.ExcelFile(f, engine=engine)
        for sheet in xl.sheet_names:
            try:
                df_s = pd.read_excel(f, sheet_name=sheet, header=None, engine=engine)
                sheet_inventory.append({
                    'file': f.name,
                    'sheet': sheet,
                    'rows': df_s.shape[0],
                    'cols': df_s.shape[1],
                    'engine': engine
                })
            except Exception as e:
                sheet_inventory.append({
                    'file': f.name,
                    'sheet': sheet,
                    'rows': -1,
                    'cols': -1,
                    'engine': engine
                })
    except Exception as e:
        sheet_inventory.append({
            'file': f.name,
            'sheet': 'ERROR: ' + str(e)[:50],
            'rows': -1,
            'cols': -1,
            'engine': engine
        })

df_sheets = pd.DataFrame(sheet_inventory)
pd.set_option('display.max_rows', 200)
print(df_sheets.to_string(index=False))
print(f"\nTotal sheets to parse: {len(df_sheets)}")

                                file                           sheet  rows  cols   engine
                 1. CIVIL WORKS.xlsx                  1. CIVIL WORKS    30     7 openpyxl
                        1.Civil.xlsx                         1.Civil    65     7 openpyxl
                    2. FLOORING.xlsx                     2. FLOORING    32     7 openpyxl
                     3. CEILING.xlsx                      3. CEILING    22     7 openpyxl
                   4. CARPENTRY.xlsx                    4. CARPENTRY    53     8 openpyxl
             5. GLASS AND METAL.xlsx              5. GLASS AND METAL    35     7 openpyxl
                    6. PAINTING.xlsx                     6. PAINTING    13     7 openpyxl
              Addendum (Facade).xlsx               Addendum (Facade)    20     6 openpyxl
                      BOQ - 01 .xlsx  Civil & ID BOQ - SA Furniture    124     6 openpyxl
                      BOQ - 02 .xlsx                  BOQ - Plumbing   140     7 openpyxl
          

## Step 3 — Deep Dive Into One Reference File

Before building the parser, we need to **fully understand the raw data structure**. Let's deeply examine `Civil & ID BOQ.xlsx` — it's a typical civil works sheet and representative of the general BOQ format.

We read it with `header=None` so we see the raw bytes exactly as they are in Excel — no header assumptions.

In [4]:
ref_file = RAW_DIR / 'Civil & ID BOQ.xlsx'
df_raw = pd.read_excel(ref_file, header=None, engine='openpyxl')

print("=" * 70)
print(f"SHAPE: {df_raw.shape}  (rows × cols)")
print("=" * 70)

print("\n--- RAW FIRST 25 ROWS ---")
print(df_raw.head(25).to_string())

print("\n" + "=" * 70)
print("--- DTYPES ---")
print(df_raw.dtypes)

print("\n" + "=" * 70)
print("--- NaN COUNT PER COLUMN ---")
print(df_raw.isnull().sum())

SHAPE: (715, 12)  (rows × cols)

--- RAW FIRST 25 ROWS ---
                                                    0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              1                            2         3                        4                 5     6    7     8      9      10   11
0      BILL OF QUANTITY FOR THE CIVIL, ID & WOOD WORKS           

## What We Observe From the Raw Output

Looking at the raw data, we can see a consistent pattern across all BOQ files:

- **Rows 0–4**: Project metadata — company name, project title, blank spacer rows. This is "header junk" we must skip.
- **Row 5** (approximately): The **actual column headers** — `S.No.`, `DESCRIPTION OF ITEM`, `QTY`, `UNIT`, `RATE (Rs.)`, `AMOUNT (Rs.)`. The exact row varies per file.
- **Row 6 onwards**: Actual data — but mixed with section headers, spec paragraphs, and blank rows.
- **Many NaN values**: Excel uses merged cells (e.g. a description spanning 3 columns). When pandas reads merged cells, only the top-left cell gets the value; the rest are NaN.

**This pattern repeats across ALL 70+ files.** Our parser must:
1. Dynamically find where the real headers start (not assume row 5)
2. Handle all the NaN noise from merged cells
3. Distinguish the 4 row types mixed in the data

## Data Issues Found — What Broke and Why

After running the parser on the 70 files, we got 44 successes and **26 failures**.
We diagnosed 4 root causes, all now fixed. A junior developer must understand these
because the same patterns will appear in any new BOQ files added to the dataset.

---

### Bug 1 — `InvalidIndexError` on `pd.concat` (crashed Cell 22)

**Root cause**: `normalize_columns()` mapped two different Excel columns to the same
standard name. For example:
- `NL-Galleria BOQ.xlsx` had two QTY columns → both became `QTY`
- `Wood BOQ.xlsx` had `RATE` (empty, col 8) AND `Rate` (with values, col 9) → both
  became `RATE`. The empty column won (first match), so all rate data was lost.

When `pd.concat()` receives DataFrames with duplicate column names it raises
`InvalidIndexError: Reindexing only valid with uniquely valued Index objects`.

**Fix**: `_dedup_columns()` — after normalization, for each duplicate, keep the column
with the most non-null values using positional iloc indexing (safe for duplicate names).

---

### Bug 2 — `S.#`, `ITEM NO`, `Item` not recognized as SNO (killed 20+ files)

**Root cause**: The SNO keyword list only contained `['s.no', 'sr.', 'sr #', 'sr#',
'item #', 'sno', 'serial']`. It missed:

| Column name in file | File(s) |
|---|---|
| `S.#` | All 15 Variation Phase files, Addendum (Facade) |
| `ITEM NO` | Civil & ID BOQ, Wood BOQ, Wardrobe BOQ, Window Sill |
| `Item` | HVAC WORKS, HVAC WORKS (2) |

When SNO is not recognized, every row gets `SNO = NaN`. Then `classify_row` sees
`sno_str = ''` → falls through to `spec_detail` for any row with a description.
Result: **0 line_items** from the entire file.

**Fix**: Expanded SNO detection with exact-match list (`'item'`, `'item no'`, `'s.#'`,
`'ref #'`) and substring list.

---

### Bug 3 — `classify_row` discarded rate rows (killed Wood BOQ, Wardrobe, Addendum)

**Root cause**: The original logic was:
```python
if not sno_str and not has_desc:  return 'empty'
if not sno_str and has_desc:      return 'spec_detail'   # ← BUG
```
The spec_detail check fired on ANY row without an SNO — including rows that
**had a rate**. In files like Wood BOQ and Wardrobe BOQ, the BOQ structure is:
```
SNO=1  | Title only (no rate)         ← section_header
SNO=NaN| Full spec + RATE=7780        ← should be line_item, was spec_detail!
```

**Fix**: Rate check inserted BEFORE spec_detail check:
```python
if not sno_str and has_rate:  return 'line_item'   # ← NEW first
if not sno_str and has_desc:  return 'spec_detail'
```

---

### Bug 4 — Variation - 2/3/4 use split rates (Material + Labor + HSE)

**Root cause**: These files have no composite `Rate` column — they break rates into
`Material Rate`, `Labor Rate`, `HSE Cost`. Our parser only looked for a single RATE
column and found nothing.

**Fix**: `normalize_columns()` now maps `Material Rate` → `MATERIAL_RATE` and
`Labor Rate` → `LABOR_RATE`. In `parse_boq_file()`, if `RATE` is all NaN but
component columns exist, we compute: `RATE = MATERIAL_RATE + LABOR_RATE + HSE_COST`.

---

### Result After Fixes

| Metric | Before | After |
|---|---|---|
| Files parsed successfully | 44 / 70 | **70 / 70** |
| Total line items extracted | ~828 | **2,125** |
| Line items with valid rates | ~700 | **1,459** |
| Files still failing | 26 | **0** |

## Step 4 — Header Row Detection

The single most important parsing challenge: **finding where the real column headers are**.

Different files put their headers on different rows (some at row 3, some at row 7, some at row 10). We can't hardcode a row number. Instead, we scan top-to-bottom looking for a row that contains BOQ keywords like `DESCRIPTION`, `QTY`, `UNIT`, `RATE`. If at least 2 keywords match → that's the header row.

In [5]:
def find_header_row(df, keywords=None):
    """
    Scan rows top-to-bottom looking for the row that contains 
    BOQ column headers like DESCRIPTION, QTY, UNIT, RATE.
    Returns the row index, or None if not found.
    
    Strategy: concatenate all non-null cell values in a row into one string,
    lowercase it, then count how many keywords appear. If >= 2 matches,
    this is likely the header row.
    """
    if keywords is None:
        keywords = ['description', 'qty', 'unit', 'rate', 'amount', 'sr', 's.no',
                    'item no', 'item', 's.#']
    
    for i, row in df.iterrows():
        row_text = ' '.join(str(v).lower() for v in row.values if pd.notna(v))
        matches = sum(1 for kw in keywords if kw in row_text)
        if matches >= 2:  # at least 2 keyword matches = likely header row
            return i
    return None


# Test on reference file
header_idx = find_header_row(df_raw)
print(f"Civil & ID BOQ.xlsx → Header row found at index: {header_idx}")
print(f"  Row content: {df_raw.iloc[header_idx].dropna().tolist()}")

print("\n" + "=" * 60)
print("Testing on 5 other files:")
print("=" * 60)

test_files = [
    'HVAC WORKS.xlsx',
    'ELECTRIC WORKS.xlsx',
    'Variation - 1.xlsx',
    'Wood BOQ.xlsx',
    'Plumbing Works.xlsx',
]

for fname in test_files:
    fpath = RAW_DIR / fname
    if not fpath.exists():
        matches = list(RAW_DIR.glob(f'*{fname.split(".")[0]}*'))
        fpath = matches[0] if matches else None
    if fpath and fpath.exists():
        try:
            engine = 'xlrd' if fpath.suffix == '.xls' else 'openpyxl'
            df_t = pd.read_excel(fpath, header=None, engine=engine)
            h = find_header_row(df_t)
            row_preview = df_t.iloc[h].dropna().tolist()[:5] if h is not None else []
            print(f"  {fpath.name:<45} → header at row {h}  |  {row_preview}")
        except Exception as e:
            print(f"  {fname:<45} → ERROR: {e}")
    else:
        print(f"  {fname:<45} → FILE NOT FOUND, skipping")


Civil & ID BOQ.xlsx → Header row found at index: 4
  Row content: ['ITEM NO', 'ITEM/DESCRIPTION', 'Material Title\nOn Drawings', 'AREACODE', 'SHEET NO CD\nxxxx-xx-xx', 'Mesuremets / Cal', 'UNIT', 'QTY', 'Rate', 'Total', 'NOTES']

Testing on 5 other files:
  HVAC WORKS.xlsx                               → header at row 4  |  ['Item', 'Description', 'Qty.', 'Unit', 'Rate (Rs.)']
  ELECTRIC WORKS.xlsx                           → header at row 4  |  ['SR. NO', 'Items Description ', 'Unit ', 'Qty. ', 'Unit Rate']
  Variation - 1.xlsx                            → header at row 4  |  ['S.No.', 'Description', 'Estimate Qty.', 'Unit', 'Material Rate']
  Wood BOQ.xlsx                                 → header at row 2  |  ['ITEM NO', 'ITEM/DESCRIPTION', 'Material Title\nOn Drawings', 'AREACODE', 'SHEET NO CD\nxxxx-xx-xx']
  Plumbing Works.xlsx                           → header at row 4  |  ['Sr#.', 'Description', 'Unit', 'Qty', 'Rate']


## Step 5 — Re-Read With Correct Header

Now that we can find the header row, let's re-read the file correctly and see the actual column names. We pass `skiprows` to `read_excel` so pandas treats that row as the header.

In [6]:
header_idx = find_header_row(df_raw)

df_with_header = pd.read_excel(
    ref_file,
    header=header_idx,
    engine='openpyxl'
)

print("Column names as read from Excel:")
print(df_with_header.columns.tolist())
print("\n" + "=" * 70)
print("--- FIRST 20 DATA ROWS ---")
print(df_with_header.head(20).to_string())

Column names as read from Excel:
['ITEM NO', 'ITEM/DESCRIPTION', 'Material Title\nOn Drawings', 'AREACODE', 'SHEET NO CD\nxxxx-xx-xx', 'Mesuremets / Cal', 'UNIT', 'QTY', 'Rate', 'Total', 'NOTES', 'Unnamed: 11']

--- FIRST 20 DATA ROWS ---
              ITEM NO                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ITEM/DESCRIPTION Material Title\nOn Drawings AREACODE

## Step 6 — Row Classification

**Critical challenge**: Not every row in a BOQ is a billable line item. We have 4 types of rows mixed together:

| Type | S.No pattern | Has Rate? | Example |
|------|-------------|-----------|--------|
| `section_header` | Integer (1, 2, 3) | No | "Dismantling Works" |
| `line_item` | Decimal (1.1, 1.2) OR letter (a, b, i) | Yes | "Brick Work 4.5 Thick" |
| `spec_detail` | NaN | No | Long description paragraph / material spec |
| `empty` | NaN | NaN | Blank spacer row |

We only want `line_item` rows. The classifier uses three signals: the S.No format, whether a rate exists, and whether there is a meaningful description.

In [7]:
def classify_row(row, sno_col, rate_col, desc_col):
    """
    Classify each row: 'empty' | 'spec_detail' | 'section_header' | 'line_item'

    Decision tree (order matters):
    1. No SNO + no desc + no rate          → empty
    2. No SNO + has rate                   → line_item
       (Wood/Wardrobe BOQ: rate on the NaN-SNO spec row)
    3. No SNO + has desc (no rate)         → spec_detail
    4. Numeric SNO (int OR decimal) + rate → line_item
    5. Numeric SNO (int OR decimal) no rate→ section_header  ← FIX
       Previously only whole integers were caught. Decimal SNOs like
       1.1, 1.2, 1.3 are TIER-1 section headers in many files.
       Without this fix, section_title never advanced correctly.
    6. Non-numeric SNO (a, b, i) Roman)   → line_item
    """
    sno  = row.get(sno_col, None)
    rate = row.get(rate_col, None)
    desc = row.get(desc_col, None)
    sno_str  = str(sno).strip() if pd.notna(sno) else ''
    has_rate = pd.notna(rate) and str(rate).strip() not in ['', '0', 'nan', 'NaN']
    has_desc = pd.notna(desc) and len(str(desc).strip()) > 3
    if not sno_str and not has_desc and not has_rate: return 'empty'
    if not sno_str and has_rate:                      return 'line_item'
    if not sno_str and has_desc:                      return 'spec_detail'
    try:
        float(sno_str)                        # succeeds for 1, 1.1, 1.2 …
        return 'line_item' if has_rate else 'section_header'
    except: pass
    if has_rate or sno_str: return 'line_item'  # a), b), i), roman nums
    return 'spec_detail'


# Apply to reference file
df_ref = df_with_header.copy()
df_ref.columns = [str(c) for c in df_ref.columns]
sno_col  = next((c for c in df_ref.columns if any(x in c.lower() for x in ['s.no','sr','sno'])), df_ref.columns[0])
rate_col = next((c for c in df_ref.columns if 'rate' in c.lower()), df_ref.columns[-2])
desc_col = next((c for c in df_ref.columns if 'desc' in c.lower() or 'item' in c.lower()), df_ref.columns[1])
print(f"Using → SNO: '{sno_col}'  RATE: '{rate_col}'  DESC: '{desc_col}'")
df_ref['ROW_TYPE'] = df_ref.apply(lambda r: classify_row(r, sno_col, rate_col, desc_col), axis=1)
print('\nRow type distribution:')
print(df_ref['ROW_TYPE'].value_counts())
for rtype in ['section_header','line_item','spec_detail','empty']:
    sub = df_ref[df_ref['ROW_TYPE']==rtype]
    print(f"\n{'='*60}\n--- TYPE: {rtype} ({len(sub)} rows) ---")
    print(sub[[sno_col,desc_col,rate_col]].head(3).to_string())


Using → SNO: 'ITEM NO'  RATE: 'Rate'  DESC: 'ITEM NO'

Row type distribution:
ROW_TYPE
line_item         631
empty              66
section_header     13
Name: count, dtype: int64

--- TYPE: section_header (13 rows) ---
    ITEM NO ITEM NO  Rate
239     3.1     3.1   NaN
262     4.1     4.1   NaN
263     4.2     4.2   NaN

--- TYPE: line_item (631 rows) ---
        ITEM NO       ITEM NO  Rate
0   Gen. \nSPEC   Gen. \nSPEC   NaN
1  MARBLE/STONE  MARBLE/STONE   NaN
2    1.0 \nSPEC    1.0 \nSPEC   NaN

--- TYPE: spec_detail (0 rows) ---
Empty DataFrame
Columns: [ITEM NO, ITEM NO, Rate]
Index: []

--- TYPE: empty (66 rows) ---
   ITEM NO ITEM NO  Rate
7      NaN     NaN   NaN
31     NaN     NaN   NaN
35     NaN     NaN   NaN


## Step 7 — Column Name Normalization

Now the hardest part: **column name normalization**. Every file uses slightly different column names for the same data:

| Standard Name | Variations Seen in Files |
|---|---|
| `SNO` | `S.No.` · `Sr. #` · `Sr#.` · `S.No` · `SNO` · `Item #` |
| `DESCRIPTION` | `DESCRIPTION OF ITEM` · `Description` · `Items Description` · `Particulars` |
| `QTY` | `QTY` · `Quantity` · `Estimate Qty` |
| `UNIT` | `UNIT` · `Units` |
| `RATE` | `RATE (Rs.)` · `Unit Rate \n(Rs.)` · `Rate` · `RATE` · `Accumulated Rate` · `Material + Labor` |
| `AMOUNT` | `AMOUNT (Rs.)` · `Amount \n(Rs.)` · `AMOUNT` · `Amount` · `Total` |

Some files also have extra columns like `Material Rate`, `Labor Rate`, `HSE Cost` — we ignore those (we only care about the final composite rate).

In [8]:
def normalize_columns(df):
    """
    Map all column name variations to standard names:
    SNO, DESCRIPTION, QTY, UNIT, RATE, AMOUNT, MATERIAL_RATE, LABOR_RATE, HSE
    
    Key fixes vs original version:
    1. SNO patterns expanded to include 's.#', 'item no', standalone 'item'.
       Many files use these (Variation Phase series, Wood BOQ, Wardrobe BOQ,
       HVAC WORKS, Civil & ID BOQ) — all returned 0 items because SNO was not
       recognized, so every row got classified as spec_detail.
    2. MATERIAL_RATE and LABOR_RATE mapped separately. Variation - 2/3/4 break
       rates into Material + Labor + HSE components with the composite Rate column
       left blank. We track components so parse_boq_file can sum them.
    3. Duplicate column resolution happens in parse_boq_file (not here), but we
       use PRIORITY ordering: 'accumulated'/'composite' beats plain 'rate', and
       we don't overwrite a higher-priority mapping with a lower one.
    
    Returns df with renamed columns.
    """
    # Priority tiers for RATE: tier 1 = specific accumulated, tier 2 = plain rate
    RATE_PRIORITY = {}
    col_map = {}
    
    for col in df.columns:
        col_clean = str(col).lower().strip().replace('\n', ' ')
        
        # SNO — expanded: s.#, item no, standalone 'item'
        if col_clean.strip() in ['item', 'item no', 'item no.', 's.#', 'ref #', 'ref#', '#', 'no.']:
            col_map[col] = 'SNO'
        elif any(x in col_clean for x in ['s.no', 'sr.', 'sr #', 'sr#', 'item #', 'sno', 'serial', 's.#', 'item no']):
            col_map[col] = 'SNO'
        
        # DESCRIPTION — must come before SNO catch-all for 'item'
        elif any(x in col_clean for x in ['description', 'items desc', 'item desc', 'particular', 'item/desc']):
            col_map[col] = 'DESCRIPTION'
        
        # QTY
        elif any(x in col_clean for x in ['qty', 'quantity', 'estimate qty']):
            col_map[col] = 'QTY'
        
        # UNIT
        elif col_clean.strip() in ['unit', 'units']:
            col_map[col] = 'UNIT'
        
        # RATE — tier 1: accumulated/composite (preferred)
        elif any(x in col_clean for x in ['accumalated rate', 'accumulated rate', 'composite',
                                           'unit rate', 'rate (rs', 'rate(rs', 'material + labor']):
            col_map[col] = 'RATE'
            RATE_PRIORITY[col] = 1
        
        # MATERIAL_RATE and LABOR_RATE — tracked separately for accumulation
        elif 'material' in col_clean and 'rate' in col_clean:
            col_map[col] = 'MATERIAL_RATE'
        elif 'labor' in col_clean and 'rate' in col_clean:
            col_map[col] = 'LABOR_RATE'
        elif col_clean.strip() in ['hse', 'hse cost']:
            col_map[col] = 'HSE_COST'
        
        # RATE — tier 2: plain 'rate' columns
        elif 'rate' in col_clean:
            col_map[col] = 'RATE'
            RATE_PRIORITY[col] = 2
        
        # AMOUNT
        elif any(x in col_clean for x in ['amount', 'total']):
            col_map[col] = 'AMOUNT'
    
    return df.rename(columns=col_map), RATE_PRIORITY


# Test on 5 different files
test_norm_files = [
    'Civil & ID BOQ.xlsx',
    'HVAC WORKS.xlsx',
    'ELECTRIC WORKS.xlsx',
    'Variation Phase - 3.xlsx',
    'Wood BOQ.xlsx',
]

print("Column normalization test (expanded SNO patterns):")
print("=" * 80)

for fname in test_norm_files:
    fpath = RAW_DIR / fname
    if not fpath.exists():
        candidates = list(RAW_DIR.glob(f'*{fname.split(".")[0]}*'))
        fpath = candidates[0] if candidates else None
    if fpath and fpath.exists():
        try:
            engine = 'xlrd' if fpath.suffix == '.xls' else 'openpyxl'
            df_t = pd.read_excel(fpath, header=None, engine=engine)
            h = find_header_row(df_t)
            if h is not None:
                df_t2 = pd.read_excel(fpath, header=h, engine=engine)
                df_t2.columns = [str(c) for c in df_t2.columns]
                original_cols = df_t2.columns.tolist()
                df_normed, _ = normalize_columns(df_t2)
                new_cols = df_normed.columns.tolist()
                dupes = [c for c in new_cols if new_cols.count(c) > 1 and c != 'Unnamed: 0']
                print(f"\n{fpath.name}:")
                print(f"  Original  : {original_cols}")
                print(f"  Normalized: {new_cols}")
                if dupes:
                    print(f"  ⚠ Duplicate cols: {list(set(dupes))} → will be resolved by keeping most-populated")
        except Exception as e:
            print(f"\n{fname}: ERROR — {e}")
    else:
        print(f"\n{fname}: FILE NOT FOUND")


Column normalization test (expanded SNO patterns):

Civil & ID BOQ.xlsx:
  Original  : ['ITEM NO', 'ITEM/DESCRIPTION', 'Material Title\nOn Drawings', 'AREACODE', 'SHEET NO CD\nxxxx-xx-xx', 'Mesuremets / Cal', 'UNIT', 'QTY', 'Rate', 'Total', 'NOTES', 'Unnamed: 11']
  Normalized: ['SNO', 'DESCRIPTION', 'Material Title\nOn Drawings', 'AREACODE', 'SHEET NO CD\nxxxx-xx-xx', 'Mesuremets / Cal', 'UNIT', 'QTY', 'RATE', 'AMOUNT', 'NOTES', 'Unnamed: 11']

HVAC WORKS.xlsx:
  Original  : ['Item', 'Description', 'Qty.', 'Unit', 'Rate (Rs.)', 'Amount (Rs.)']
  Normalized: ['SNO', 'DESCRIPTION', 'QTY', 'UNIT', 'RATE', 'AMOUNT']

ELECTRIC WORKS.xlsx:
  Original  : ['SR. NO', 'Items Description ', 'Unit ', 'Qty. ', 'Unit Rate', 'Amount']
  Normalized: ['SNO', 'DESCRIPTION', 'UNIT', 'QTY', 'RATE', 'AMOUNT']

Variation Phase - 3.xlsx:
  Original  : ['S.#', 'Description', 'Unit', 'Qty.', ' Rate', 'Amount']
  Normalized: ['SNO', 'DESCRIPTION', 'UNIT', 'QTY', 'RATE', 'AMOUNT']

Wood BOQ.xlsx:
  Original  : 

## Step 8 — Context-Aware Description Builder

**The core data loss problem**: The parser was processing rows independently.
It kept `BRICKWORK 4 1/2"` (RATE=310) but silently discarded the full
specification paragraph that explained *what* was priced and *how*.

**3-tier BOQ structure:**
```
TIER 1 │ SNO=1.2  │ DESC='BRICKWORK'                    │ RATE=—  ← section_header
TIER 2 │ SNO=NaN  │ DESC='Providing and laying brick     │ RATE=—  ← spec_detail
       │          │  masonry using first-class bricks,   │         ← WAS LOST
       │          │  1:4 mortar, soaking, curing…'       │
TIER 3 │ SNO=1    │ DESC='BRICKWORK 4 1/2"'              │ RATE=310 ← line_item ✓
TIER 3 │ SNO=2    │ DESC='BRICKWORK 9"'                   │ RATE=510 ← line_item ✓
```

**Why this kills RAG**: A query like *'brick masonry 1:4 mortar rate'* will
never retrieve `'BRICKWORK 4 1/2"'` because those words aren't in the short
label. The full spec from TIER 2 **must** be embedded alongside the rate.

**Solution — stateful context propagation**:
Process rows sequentially with a state machine. Carry TIER-1 and TIER-2
context forward into every TIER-3 line item. Both sub-items (4 1/2" and 9")
share the same spec — state resets only when a new TIER-1 header appears.

**New columns produced:**

| Column | Content | Used for |
|---|---|---|
| `DESCRIPTION_SHORT` | Short label on rate row | Display |
| `SECTION_TITLE` | TIER-1 parent category | Metadata / facet |
| `SPEC_TEXT` | Full TIER-2 specification | Retrieved context |
| `DESCRIPTION_FULL` | `SECTION — SPEC \| SHORT` | **RAG embedding input** |

In [9]:
def build_full_desc(section_title, spec_text, desc_short):
    """
    Assemble the rich text for RAG embedding from the 3 tiers.

    Examples:
      All tiers:     'BRICKWORK — Providing and laying brick masonry... | BRICKWORK 4 1/2"'
      No spec:       'BRICKWORK | BRICKWORK 4 1/2"'
      No title:      'Providing and laying... | BRICKWORK 4 1/2"'
      Neither:       'BRICKWORK 4 1/2"'
    """
    desc_short = str(desc_short).strip() if pd.notna(desc_short) else ''
    front = []
    if section_title and str(section_title).strip() not in ['', desc_short]:
        front.append(str(section_title).strip())
    if spec_text and str(spec_text).strip():
        front.append(str(spec_text).strip())
    return (' — '.join(front) + ' | ' + desc_short) if front else desc_short


def extract_with_context(df):
    """
    Stateful sequential pass over one normalised sheet DataFrame.

    State maintained between rows:
        section_title   — set by TIER-1 (section_header) rows
        spec_paragraphs — accumulated from TIER-2 (spec_detail) rows,
                          reset when a new TIER-1 header appears

    On TIER-3 (line_item):
        snapshot state → attach to record → DO NOT reset
        (next sub-item shares same section + spec)

    Edge cases:
        Multiple spec paragraphs → joined with double newline
        No preceding spec        → SPEC_TEXT = None
        Empty rows               → skip, preserve state
        Wood BOQ format          → NaN-SNO rate row IS the line_item;
                                   SECTION_TITLE from numeric row above
    """
    for col in ['SNO', 'RATE', 'DESCRIPTION']:
        if col not in df.columns: df[col] = np.nan

    section_title   = None
    spec_paragraphs = []
    records         = []

    for _, row in df.iterrows():
        rtype = classify_row(row, 'SNO', 'RATE', 'DESCRIPTION')
        desc  = str(row.get('DESCRIPTION', '')).strip() if pd.notna(row.get('DESCRIPTION')) else ''

        if rtype == 'section_header':
            if desc: section_title = desc
            spec_paragraphs = []             # new section → reset spec

        elif rtype == 'spec_detail':
            if len(desc) > 10: spec_paragraphs.append(desc)

        elif rtype == 'line_item':
            spec_text = '\n\n'.join(spec_paragraphs) if spec_paragraphs else None
            records.append({
                'SNO':               row.get('SNO'),
                'DESCRIPTION_SHORT': desc,
                'SECTION_TITLE':     section_title,
                'SPEC_TEXT':         spec_text,
                'DESCRIPTION_FULL':  build_full_desc(section_title, spec_text, desc),
                'QTY':               row.get('QTY'),
                'UNIT':              row.get('UNIT'),
                'RATE':              row.get('RATE'),
                'AMOUNT':            row.get('AMOUNT'),
            })
        # 'empty' → skip, preserve state

    return records


def _dedup_columns(df):
    """
    Resolve duplicate column names from normalization.
    Keeps the column with the most non-null values.
    """
    if not df.columns.duplicated().any(): return df
    keep, seen = [], {}
    for i, col in enumerate(df.columns):
        if col not in seen: seen[col]=i; keep.append(i)
        else:
            prev = seen[col]
            if df.iloc[:,i].notna().sum() > df.iloc[:,prev].notna().sum():
                keep[keep.index(prev)]=i; seen[col]=i
    return df.iloc[:,sorted(keep)]


def parse_boq_file(filepath):
    """
    Parse one BOQ file through the full pipeline:
      find_header_row → normalize_columns → _dedup_columns
      → (accumulate split rates) → extract_with_context
    Tags every record with SOURCE_FILE and SHEET_NAME.
    Returns (DataFrame, None) on success or (None, error_str) on failure.
    """
    filepath    = Path(filepath)
    engine      = 'xlrd' if filepath.suffix == '.xls' else 'openpyxl'
    all_records = []
    try:
        xl = pd.ExcelFile(filepath, engine=engine)
    except Exception as e:
        return None, f"Could not open: {e}"

    for sheet in xl.sheet_names:
        try:
            df_raw = pd.read_excel(filepath, sheet_name=sheet, header=None, engine=engine)
            if df_raw.shape[0] < 5: continue
            h = find_header_row(df_raw)
            if h is None: continue
            df = pd.read_excel(filepath, sheet_name=sheet, header=h, engine=engine)
            df.columns = [str(c) for c in df.columns]
            df, _ = normalize_columns(df)
            df    = _dedup_columns(df)
            if 'DESCRIPTION' not in df.columns: continue
            # Accumulate split rates (Variation - 2/3/4 style)
            if 'RATE' not in df.columns or df['RATE'].isna().all():
                comps = [c for c in ['MATERIAL_RATE','LABOR_RATE','HSE_COST'] if c in df.columns]
                if comps:
                    df['RATE'] = df[comps].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
            if 'RATE' not in df.columns: continue
            records = extract_with_context(df)
            for r in records:
                r['SOURCE_FILE'] = filepath.name
                r['SHEET_NAME']  = sheet
            all_records.extend(records)
        except Exception: continue

    if not all_records: return None, "No line items extracted"
    return pd.DataFrame(all_records), None


# ── Demo: the file that revealed the context-loss problem ─────────────────
print("Demo — 1. CIVIL WORKS.xlsx (context propagation)")
print("=" * 65)
demo_df, _ = parse_boq_file(RAW_DIR / '1. CIVIL WORKS.xlsx')
if demo_df is not None:
    for _, r in demo_df.iterrows():
        if pd.notna(r['RATE']):
            print(f"\n  SHORT:  {r['DESCRIPTION_SHORT']}")
            print(f"  SECT:   {r['SECTION_TITLE']}")
            spec_p = str(r['SPEC_TEXT'] or '')[:85].replace('\n',' ')
            print(f"  SPEC:   {spec_p}{'...' if len(spec_p)==85 else ''}")
            print(f"  RATE:   {r['RATE']}  UNIT: {r['UNIT']}")
            print(f"  FULL:   {str(r['DESCRIPTION_FULL'])[:100]}")


Demo — 1. CIVIL WORKS.xlsx (context propagation)

  SHORT:  BRICKWORK 4 1/2"
  SECT:   BRICKWORK
  SPEC:   Providing and laying brick masonry using first-class, laboratory-tested bricks having...
  RATE:   310.0  UNIT: Sft
  FULL:   BRICKWORK — Providing and laying brick masonry using first-class, laboratory-tested bricks having a 

  SHORT:  BRICKWORK 9"
  SECT:   BRICKWORK
  SPEC:   Providing and laying brick masonry using first-class, laboratory-tested bricks having...
  RATE:   510.0  UNIT: Sft
  FULL:   BRICKWORK — Providing and laying brick masonry using first-class, laboratory-tested bricks having a 

  SHORT:  INTERNAL PLASTER
  SECT:   PLASTER
  SPEC:   Providing and applying cement–sand plaster of 10mm to 16mm thickness in cement mortar...
  RATE:   90.0  UNIT: Sft
  FULL:   PLASTER — Providing and applying cement–sand plaster of 10mm to 16mm thickness in cement mortar (Bes

  SHORT:  LINTELS [4.5"X6"X4']
  SECT:   REINFORCED CEMENT CONCRETE
  SPEC:   Providing, mixing, placi

## Step 9 — Run Parser on ALL 70 Files

Let's run the parser on all files and collect the results.
`Plumbing Works.xlsx` will return 0 — it is an **unpriced tender**
(all RATE cells blank, sent to contractors for pricing). That is correct.

In [10]:
all_results = []
errors      = []

files = [f for f in RAW_DIR.glob('*')
         if f.suffix in ['.xlsx', '.xls']
         and 'Zone.Identifier' not in f.name]

print(f"Processing {len(files)} files...\n")

for filepath in sorted(files):
    df, err = parse_boq_file(filepath)
    if err:
        errors.append({'file': filepath.name, 'error': err})
        print(f"  ✗ {filepath.name:50s} | {err}")
    else:
        all_results.append(df)
        print(f"  ✓ {filepath.name:50s} | {len(df)} rows")

print(f"\n{'='*60}")
print(f"Successfully parsed: {len(all_results)} / {len(files)} files")
print(f"No rates (expected): {len(errors)} file(s)")


Processing 70 files...

  ✓ 1. CIVIL WORKS.xlsx                                | 5 rows
  ✓ 1.Civil.xlsx                                       | 38 rows
  ✓ 2. FLOORING.xlsx                                   | 9 rows
  ✓ 3. CEILING.xlsx                                    | 5 rows
  ✓ 4. CARPENTRY.xlsx                                  | 21 rows
  ✓ 5. GLASS AND METAL.xlsx                            | 9 rows
  ✓ 6. PAINTING.xlsx                                   | 2 rows
  ✓ Addendum (Facade).xlsx                             | 11 rows
  ✓ BOQ - 01 .xlsx                                     | 41 rows
  ✓ BOQ - 02 .xlsx                                     | 29 rows
  ✓ BOQ - 03 .xlsx                                     | 10 rows
  ✓ BOQ - 05 .xlsx                                     | 228 rows
  ✓ BOQ - 12.xlsx                                      | 2 rows
  ✓ BOQ - 14 .xlsx                                     | 17 rows
  ✓ BOQ-06.xlsx                                        | 32 rows
  ✓ CC

In [11]:
df_all = pd.concat(all_results, ignore_index=True)
print(f"Total rows combined: {df_all.shape}")
print(f"\nColumns: {df_all.columns.tolist()}")
print(f"\nNull counts:\n{df_all.isnull().sum()}")
print(f"\nRows per source file:")
print(df_all['SOURCE_FILE'].value_counts().to_string())
print(f"\n{'='*70}")
print("SAMPLE — first record with SPEC_TEXT (context propagation working):")
s = df_all[df_all['SPEC_TEXT'].notna()].iloc[0]
print(f"  SOURCE_FILE:       {s['SOURCE_FILE']}")
print(f"  SECTION_TITLE:     {s['SECTION_TITLE']}")
print(f"  SPEC_TEXT (start): {str(s['SPEC_TEXT'])[:100]}...")
print(f"  DESCRIPTION_SHORT: {s['DESCRIPTION_SHORT']}")
print(f"  DESCRIPTION_FULL:  {str(s['DESCRIPTION_FULL'])[:110]}...")
print(f"  RATE: {s['RATE']}   UNIT: {s['UNIT']}")


Total rows combined: (1852, 11)

Columns: ['SNO', 'DESCRIPTION_SHORT', 'SECTION_TITLE', 'SPEC_TEXT', 'DESCRIPTION_FULL', 'QTY', 'UNIT', 'RATE', 'AMOUNT', 'SOURCE_FILE', 'SHEET_NAME']

Null counts:
SNO                   422
DESCRIPTION_SHORT       0
SECTION_TITLE         238
SPEC_TEXT            1275
DESCRIPTION_FULL        0
QTY                   380
UNIT                  270
RATE                  355
AMOUNT               1384
SOURCE_FILE             0
SHEET_NAME              0
dtype: int64

Rows per source file:
SOURCE_FILE
Civil & ID BOQ.xlsx                     631
BOQ - 05 .xlsx                          228
Electric BOQ.xlsx                       106
ELECTRIC WORKS.xlsx                      78
Nishat Linen (Civil ,ID & MEP ).xlsx     66
ID Works (G.F).xlsx                      46
BOQ - 01 .xlsx                           41
ID Works (F.F).xlsx                      40
1.Civil.xlsx                             38
NL-Galleria BOQ.xlsx                     33
BOQ-06.xlsx                  

## Step 10 — Rate Column Cleaning

Now let's clean the `RATE` column. Excel rates come in many dirty formats:

- `"Rs. 1,35,000"` — Pakistani currency format with prefix
- `"1,500.00"` — comma-separated thousands
- `315.0` — already a float (from numeric cells)
- `"OFM"` — "Owner Furnished Material" (no rate, owner supplies)
- `NaN` — blank (item not yet priced, often in tender documents)

We clean them all to float or None.

In [12]:
def clean_rate(val):
    """
    Convert messy rate values to float.
    
    Handles:
    - "Rs. 1,35,000"  → 135000.0
    - "1,500.00"       → 1500.0
    - 315.0            → 315.0
    - "OFM" / "N/A"   → None  (non-numeric text)
    - NaN              → None
    - 0                → None  (zero rates are useless for pricing)
    
    Returns float or None.
    """
    if pd.isna(val):
        return None
    s = str(val).strip()
    # Remove currency symbols, text prefixes
    s = re.sub(r'[Rrs\.\s]', '', s)   # Remove Rs.
    s = re.sub(r'[^\d\.,]', '', s)     # Keep only digits, comma, dot
    s = s.replace(',', '')              # Remove commas
    try:
        f = float(s)
        return f if f > 0 else None
    except:
        return None


# Apply
df_all['RATE_CLEAN'] = df_all['RATE'].apply(clean_rate)
df_all['AMOUNT_CLEAN'] = df_all['AMOUNT'].apply(clean_rate)

# Stats
total = len(df_all)
has_rate = df_all['RATE_CLEAN'].notna().sum()
print(f"Total rows:                  {total}")
print(f"Rows with valid rate:        {has_rate} ({100*has_rate/total:.1f}%)")
print(f"Rows WITHOUT rate (blank):   {total - has_rate} ({100*(total-has_rate)/total:.1f}%)")
print(f"\nRate statistics:")
print(df_all['RATE_CLEAN'].describe())

Total rows:                  1852
Rows with valid rate:        1459 (78.8%)
Rows WITHOUT rate (blank):   393 (21.2%)

Rate statistics:
count    1.459000e+03
mean     4.976552e+14
std      3.314446e+15
min      1.500000e+01
25%      3.225000e+03
50%      1.700000e+04
75%      1.400000e+05
max      6.333333e+16
Name: RATE_CLEAN, dtype: float64


In [13]:
def get_work_category(filename):
    """
    Assign a work category from the source filename.
    Mirrors the categorize_file() logic from the inventory step
    but simplified to fewer buckets suitable for the RAG knowledge base.
    """
    f = filename.lower()
    if any(x in f for x in ['civil', 'flooring', 'ceiling', 'carpentry', 'painting', 'glass', 'id works', 'nl-galleria', 'nishat']):
        return 'civil_id'
    elif any(x in f for x in ['hvac', 'air']):
        return 'hvac'
    elif any(x in f for x in ['electric', 'switchgear', 'wiring', 'cable', 'earthing', 'fitting', 'cctv', 'data', 'sound', 'pa &', 'shop']):
        return 'electrical_elv'
    elif 'plumbing' in f:
        return 'plumbing'
    elif 'fire' in f or f == 'fa.xlsx':
        return 'fire_fighting'
    elif 'variation' in f:
        return 'variation'
    elif any(x in f for x in ['wood', 'wardrobe', 'window', 'stair', 'addendum', 'boq']):
        return 'special_works'
    return 'other'


df_all['WORK_CATEGORY'] = df_all['SOURCE_FILE'].apply(get_work_category)
print("Line items per work category (before rate filter):")
print(df_all['WORK_CATEGORY'].value_counts())

Line items per work category (before rate filter):
WORK_CATEGORY
civil_id          905
special_works     388
electrical_elv    291
variation         180
other              38
hvac               29
fire_fighting      21
Name: count, dtype: int64


## Step 11 — Final Dataset

Keep only rows where we have a valid rate. These are the rows we can actually use for pricing lookups in the RAG system. Rows without rates are unpriced tenders — interesting for description completeness but useless for rate retrieval.

In [ ]:
df_final = df_all[df_all['RATE_CLEAN'].notna()].copy().reset_index(drop=True)

# Drop columns not needed in the final knowledge base
df_final = df_final.drop(columns=['RATE_CLEAN', 'AMOUNT'])

print(f"Final dataset shape: {df_final.shape}")
print(f"\nSample rows:")
print(df_final[['DESCRIPTION_SHORT','SECTION_TITLE','QTY','UNIT',
                 'RATE','SOURCE_FILE','WORK_CATEGORY']].head(15).to_string())

# Context coverage stats
has_spec = df_final['SPEC_TEXT'].notna().sum()
has_sect = df_final['SECTION_TITLE'].notna().sum()
both     = (df_final['SPEC_TEXT'].notna() & df_final['SECTION_TITLE'].notna()).sum()
print(f"\nContext coverage (critical for RAG quality):")
print(f"  Rows with SECTION_TITLE:       {has_sect} ({100*has_sect/len(df_final):.1f}%)")
print(f"  Rows with SPEC_TEXT:           {has_spec} ({100*has_spec/len(df_final):.1f}%)")
print(f"  Rows with BOTH (richest):      {both} ({100*both/len(df_final):.1f}%)")

df_final.to_csv(PROCESSED_DIR / 'boq_line_items.csv', index=False)
print(f"\n✓ Saved to data/processed/boq_line_items.csv")
print(f"  Rows:    {len(df_final)}")
print(f"  Columns: {df_final.columns.tolist()}")


In [ ]:
print('='*60)
print('FINAL DATASET SUMMARY')
print('='*60)
print(f'Total line items with rates:  {len(df_final)}')
print(f'Unique source files:          {df_final["SOURCE_FILE"].nunique()}')
print(f'\nWork categories:\n{df_final["WORK_CATEGORY"].value_counts()}')
print(f'\nTop 10 units used:\n{df_final["UNIT"].value_counts().head(10)}')
print(f'\n{"="*60}')
print('DESCRIPTION_FULL SAMPLES — what gets embedded in RAG:')
print('='*60)
for cat in df_final['WORK_CATEGORY'].unique():
    sub = df_final[(df_final['WORK_CATEGORY']==cat) & df_final['SPEC_TEXT'].notna()]
    if sub.empty: sub = df_final[df_final['WORK_CATEGORY']==cat]
    row = sub.iloc[0]
    full = str(row['DESCRIPTION_FULL'])[:130]
    print(f'\n--- {cat.upper()} ---')
    print(f'  {full}...')
    print(f'  Rate: {row["RATE_CLEAN"]:,.0f}  /  {row["UNIT"]}')


## Summary — What We Built

### Final Output Schema

| Column | Description | RAG Role |
|---|---|---|
| `DESCRIPTION_FULL` | SECTION_TITLE — SPEC_TEXT \| DESCRIPTION_SHORT | **Primary embedding input** |
| `DESCRIPTION_SHORT` | Short label on the rate row | Display / filtering |
| `SECTION_TITLE` | TIER-1 parent category | Metadata / faceted search |
| `SPEC_TEXT` | Full TIER-2 specification paragraph | Retrieved verbatim in answer |
| `RATE_CLEAN` | Cleaned float rate (PKR) | Returned in RAG answer |
| `UNIT` | Unit of measure | Returned in RAG answer |
| `WORK_CATEGORY` | Trade category | Pre-filter / routing |
| `SOURCE_FILE` | Origin Excel file | Traceability |

### Parsing Challenges Solved

| Challenge | Solution |
|---|---|
| Header row is not always row 0 | `find_header_row()` — keyword scan |
| Column names vary per file | `normalize_columns()` — keyword mapping |
| Duplicate column names → concat crash | `_dedup_columns()` — keep most-populated |
| Decimal SNO rows (1.1, 1.2) misclassified | Fixed: any numeric SNO + no rate = section_header |
| Rich spec lost, only short label kept | `extract_with_context()` — stateful TIER propagation |
| Variation files use split rates | Sum MATERIAL_RATE + LABOR_RATE + HSE_COST → RATE |

### What's Next — Notebook 02
Takes `data/processed/boq_line_items.csv` and:
1. Embeds `DESCRIPTION_FULL` (full context string) using an embedding model
2. Stores vectors + metadata in ChromaDB / FAISS
3. At query time: embed the user's item description → retrieve top-k matches
   → return RATE_CLEAN + UNIT + SPEC_TEXT for the matched items